In [1]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 점검 (없어도 노트북은 폴백으로 끝까지 진행)
# ─────────────────────────────────────────────
# 설치가 필요하면 아래 주석을 해제하세요.
!pip install beautifulsoup4 requests -q
!pip install playwright -q && playwright install chromium

import warnings, time, re
import pandas as pd
warnings.filterwarnings("ignore")

# 1) BeautifulSoup — 오늘 핵심 파싱 도구 (대부분 D+010에서 설치됨)
try:
    from bs4 import BeautifulSoup
    HAS_BS4 = True
except ImportError:
    HAS_BS4 = False

# 2) requests — 네트워크 요청용 (없거나 막혀도 폴백으로 진행)
try:
    import requests
    HAS_REQUESTS = True
except ImportError:
    HAS_REQUESTS = False

# 3) Playwright — 브라우저 자동화 (Part 3에서 사용; 없으면 내장 HTML로 폴백)
try:
    import playwright          # 설치 여부만 확인 (실제 import는 함수 안에서)
    HAS_PLAYWRIGHT = True
except ImportError:
    HAS_PLAYWRIGHT = False

print("BeautifulSoup :", HAS_BS4)
print("requests      :", HAS_REQUESTS)
print("Playwright    :", HAS_PLAYWRIGHT, "  (없어도 내장 HTML로 학습 목표 달성)")

|                                                                                |   0% of 183.6 MiB
|■■■■■■■■                                                                        |  10% of 183.6 MiB
|■■■■■■■■■■■■■■■■                                                                |  20% of 183.6 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■                                                        |  30% of 183.6 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                                |  40% of 183.6 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                        |  50% of 183.6 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                |  60% of 183.6 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                        |  70% of 183.6 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                |  80% of 183.6 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■        |  90% of 

In [2]:
# ─────────────────────────────────────────────
# 모두마켓 신상품 페이지 — 두 버전의 HTML을 노트북 안에 넣어 둡니다(네트워크 불필요).
# ─────────────────────────────────────────────

# (1) 서버가 보내준 그대로 = requests가 받는 것. 상품은 아직 '비어 있음'.
STATIC_HTML = """<!DOCTYPE html>
<html lang="ko">
<head><title>모두마켓 - 신상품</title></head>
<body>
  <h1>모두마켓 신상품</h1>
  <div id="product-list">
    <!-- 상품은 JavaScript가 /api/products 에서 받아와 여기에 그립니다 -->
    <p class="loading">상품을 불러오는 중...</p>
  </div>
  <script src="/static/render_products.js"></script>
</body>
</html>"""

# (2) 브라우저가 JS를 실행해 상품을 다 그린 뒤 = 화면에서 우리가 보는 것.
#     (현장처럼 '오염'을 일부러 심어 둡니다: 공백·결측 카테고리·가격 표기 혼재)
RENDERED_HTML = """<!DOCTYPE html>
<html lang="ko">
<head><title>모두마켓 - 신상품</title></head>
<body>
  <h1>모두마켓 신상품</h1>
  <div id="product-list">
    <div class="product" data-id="P-2001">
      <span class="name">무선 이어폰</span><span class="category">가전</span><span class="price">89,000원</span>
    </div>
    <div class="product" data-id="P-2002">
      <span class="name">  블루투스 스피커  </span><span class="category">가전</span><span class="price">49,000원</span>
    </div>
    <div class="product" data-id="P-2003">
      <span class="name">노트북 거치대</span><span class="price">29,000원</span>
    </div>
    <div class="product" data-id="P-2004">
      <span class="name">기계식 키보드</span><span class="category">가전</span><span class="price">₩119000</span>
    </div>
    <div class="product" data-id="P-2005">
      <span class="name">USB-C 충전기</span><span class="category">가전</span><span class="price">19000</span>
    </div>
    <div class="product" data-id="P-2006">
      <span class="name">보조배터리</span><span class="category">잡화</span><span class="price">39,000원</span>
    </div>
    <div class="product" data-id="P-2007">
      <span class="name">스마트워치</span><span class="category">가전</span><span class="price">159,000원</span>
    </div>
    <div class="product" data-id="P-2008">
      <span class="name">액션캠</span><span class="category">가전</span><span class="price">229,000원</span>
    </div>
  </div>
  <script src="/static/render_products.js"></script>
</body>
</html>"""

print("두 버전 HTML 준비 완료")
print("STATIC_HTML 길이  :", len(STATIC_HTML), "자")
print("RENDERED_HTML 길이:", len(RENDERED_HTML), "자  ← 상품이 들어가 더 깁니다")

두 버전 HTML 준비 완료
STATIC_HTML 길이  : 302 자
RENDERED_HTML 길이: 1448 자  ← 상품이 들어가 더 깁니다


In [3]:
# 예제: 정적 HTML에서 상품을 찾아보자 — D+010 방식 그대로
soup_static = BeautifulSoup(STATIC_HTML, "html.parser")
products_found = soup_static.select(".product")   # 상품 div 찾기

print("정적 HTML에서 찾은 상품 수:", len(products_found))   # 0
print("#product-list 안의 글자:", repr(soup_static.select_one("#product-list").get_text(strip=True)))

정적 HTML에서 찾은 상품 수: 0
#product-list 안의 글자: '상품을 불러오는 중...'


In [4]:
# 예제: 데이터가 정말 소스에 없는지 직접 검색 — '무선 이어폰'이 STATIC_HTML에 있나?
print("'무선 이어폰'이 정적 HTML에 있나? :", "무선 이어폰" in STATIC_HTML)
print("'무선 이어폰'이 렌더링 HTML에 있나?:", "무선 이어폰" in RENDERED_HTML)
print()
print("→ 화면에 보이는 상품명조차 '서버가 준 HTML'에는 존재하지 않습니다.")

'무선 이어폰'이 정적 HTML에 있나? : False
'무선 이어폰'이 렌더링 HTML에 있나?: True

→ 화면에 보이는 상품명조차 '서버가 준 HTML'에는 존재하지 않습니다.


DOM
정의 : 문서 객체 모델(The Document Object Model, 이하 DOM) 은 HTML, XML 문서의 프로그래밍 interface 이다.

용도 : DOM은 문서의 구조화된 표현(structured representation)을 제공하며 프로그래밍 언어가 DOM 구조에 접근할 수 있는 방법을 제공하여 그들이 문서 구조, 스타일, 내용 등을 변경할 수 있게 돕는다.

구조 : nodes와 objects로 구성된다.

발전사항 : 초창기 DOM은 JS와 밀접하게 연관되어 있었지만 지금은 분리해서 각각 발전되었다. 따라서 페이지 콘텐츠들은 DOM에 저장되고 이를 JS로 접근 및 조작하는 방식이 대표적이다.
API (web or XML page) = DOM + JS (scripting language)

In [5]:
# 예제: '렌더링된 DOM'만 있으면, 파싱은 D+010과 똑같다 — 상품을 표로 추출
def parse_products(html):
    """렌더링된 HTML에서 상품을 뽑아 DataFrame으로. (D+010 파싱과 동일한 방식)"""
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for item in soup.select(".product"):
        name_el = item.select_one(".name")
        cat_el = item.select_one(".category")     # 없을 수도 있음(결측!)
        price_el = item.select_one(".price")
        rows.append({
            "product_id": item.get("data-id"),
            "name": name_el.get_text() if name_el else None,
            "category": cat_el.get_text(strip=True) if cat_el else None,
            "price_raw": price_el.get_text(strip=True) if price_el else None,
        })
    return pd.DataFrame(rows)

products = parse_products(RENDERED_HTML)
print("수집된 상품 수:", len(products))
display(products)

수집된 상품 수: 8


,product_id,name,category,price_raw
0,P-2001,무선 이어폰,가전,"89,000원"
1,P-2002,블루투스 스피커,가전,"49,000원"
2,P-2003,노트북 거치대,NaN,"29,000원"
3,P-2004,기계식 키보드,가전,₩119000
4,P-2005,USB-C 충전기,가전,19000
5,P-2006,보조배터리,잡화,"39,000원"
6,P-2007,스마트워치,가전,"159,000원"
7,P-2008,액션캠,가전,"229,000원"


In [6]:
# 예제: 수집 데이터에 랭글링 재적용 (D+003 결측 + D+005 문자열/정규식)
clean = products.copy()
clean["name"] = clean["name"].str.strip()                       # D+005: 앞뒤 공백 제거
clean["category"] = clean["category"].fillna("미분류")           # D+003: 결측 대체
clean["price"] = clean["price_raw"].str.replace(r"[^0-9]", "", regex=True).astype(int)  # D+005: 숫자만

products_final = clean[["product_id", "name", "category", "price"]]
display(products_final)
print("category 결측 처리 후:", products_final["category"].value_counts().to_dict())
print("평균 가격:", int(products_final["price"].mean()), "원")

,product_id,name,category,price
0,P-2001,무선 이어폰,가전,89000
1,P-2002,블루투스 스피커,가전,49000
2,P-2003,노트북 거치대,미분류,29000
3,P-2004,기계식 키보드,가전,119000
4,P-2005,USB-C 충전기,가전,19000
5,P-2006,보조배터리,잡화,39000
6,P-2007,스마트워치,가전,159000
7,P-2008,액션캠,가전,229000


category 결측 처리 후: {'가전': 6, '미분류': 1, '잡화': 1}
평균 가격: 91500 원


In [7]:
# 예제: 데이터가 <script> 안 JSON으로 함께 온 경우 — 브라우저 없이 추출 (D+005 정규식 복습)
import json
SCRIPT_JSON_HTML = """<html><body><div id="app"></div>
<script>window.__DATA__ = {"products":[{"name":"무선 이어폰","price":89000},{"name":"액션캠","price":229000}]};</script>
</body></html>"""

m = re.search(r"window\.__DATA__\s*=\s*(\{.*\});", SCRIPT_JSON_HTML)   # script 안 JSON만 뽑기
data = json.loads(m.group(1))
df_from_script = pd.DataFrame(data["products"])
print("화면용 #app 은 비어 있지만, 소스의 script JSON에는 데이터가 있었습니다:")
display(df_from_script)

화면용 #app 은 비어 있지만, 소스의 script JSON에는 데이터가 있었습니다:


,name,price
0,무선 이어폰,89000
1,액션캠,229000


In [30]:
# 스스로 해보자! (2)
# 아래 주석(#)을 지우고 빈칸(___)을 채워보세요.

# 1) 가전 상품만
가전 = products_final[products_final["category"] == "가전"]
print("가전 상품 수:", len(가전))

# 2) 가장 비싼 상품
top = products_final.sort_values("price", ascending=False).iloc[0]
print("최고가 상품:", top["name"], top["price"], "원")

가전 상품 수: 6
최고가 상품: 액션캠 229000 원


In [38]:
# [브라우저 설치 필요: playwright install chromium]
import concurrent.futures

def _render_sync(url, wait_selector, timeout_ms=5000):
    import asyncio, sys
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    from playwright.sync_api import sync_playwright
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto(url, timeout=timeout_ms)
        page.wait_for_selector(wait_selector, timeout=timeout_ms)
        html = page.content()
        browser.close()
    return html

try:
    with concurrent.futures.ThreadPoolExecutor() as ex:
        rendered = ex.submit(
            _render_sync, "https://quotes.toscrape.com/js/", ".quote", 30000  # 타임아웃 30초로 상향
        ).result()
    source_used = "playwright(실제 브라우저)"
except Exception as e:
    rendered = RENDERED_HTML
    source_used = f"폴백(내장 RENDERED_HTML) — 사유: {type(e).__name__}: {e!r}"

print("렌더링 소스:", source_used)
print("받은 HTML 길이:", len(rendered), "자")

렌더링 소스: playwright(실제 브라우저)
받은 HTML 길이: 8940 자


In [39]:
# 예제: 렌더링된 DOM(rendered)을 파싱 — Part 2의 함수를 그대로 재사용
# 폴백이면 모두마켓 상품 구조이므로 parse_products로, 실제 quotes면 .quote 구조로 분기
soup_r = BeautifulSoup(rendered, "html.parser")

if soup_r.select(".product"):          # 폴백(모두마켓 내장 HTML)인 경우
    result = parse_products(rendered)
    print("파싱 결과(상품):", len(result), "행")
    display(result.head(3))
else:                                   # 실제 quotes 사이트가 렌더링된 경우
    quotes = [q.select_one(".text").get_text(strip=True) for q in soup_r.select(".quote")]
    print("파싱 결과(인용문):", len(quotes), "개")
    for t in quotes[:3]:
        print("-", t[:50], "...")

파싱 결과(인용문): 10 개
- “The world as we have created it is a process of o ...
- “It is our choices, Harry, that show what we truly ...
- “There are only two ways to live your life. One is ...


In [40]:
# 예제: 렌더링된 DOM(rendered)을 파싱 — Part 2의 함수를 그대로 재사용
# 폴백이면 모두마켓 상품 구조이므로 parse_products로, 실제 quotes면 .quote 구조로 분기
soup_r = BeautifulSoup(rendered, "html.parser")

if soup_r.select(".product"):          # 폴백(모두마켓 내장 HTML)인 경우
    result = parse_products(rendered)
    print("파싱 결과(상품):", len(result), "행")
    display(result.head(3))
else:                                   # 실제 quotes 사이트가 렌더링된 경우
    quotes = [q.select_one(".text").get_text(strip=True) for q in soup_r.select(".quote")]
    print("파싱 결과(인용문):", len(quotes), "개")
    for t in quotes[:3]:
        print("-", t[:50], "...")

파싱 결과(인용문): 10 개
- “The world as we have created it is a process of o ...
- “It is our choices, Harry, that show what we truly ...
- “There are only two ways to live your life. One is ...


In [41]:
# 예제: 같은 페이지, '정적 vs 렌더링'의 수집 결과를 숫자로 비교
n_static = len(BeautifulSoup(STATIC_HTML, "html.parser").select(".product"))
n_rendered = len(BeautifulSoup(RENDERED_HTML, "html.parser").select(".product"))
print(f"requests만 받았을 때 (정적)  -> 상품 {n_static}개")
print(f"렌더링된 DOM을 봤을 때       -> 상품 {n_rendered}개")
print("-> 같은 페이지인데 '언제의 HTML이냐'가 수집 성패를 가릅니다. 그래서 도구 선택이 중요합니다.")

requests만 받았을 때 (정적)  -> 상품 0개
렌더링된 DOM을 봤을 때       -> 상품 8개
-> 같은 페이지인데 '언제의 HTML이냐'가 수집 성패를 가릅니다. 그래서 도구 선택이 중요합니다.


In [42]:
# 스스로 해보자! (3)
# 아래 주석(#)을 지우고 실험해보세요. (브라우저가 없으면 폴백 메시지가 보입니다)

try:
    test = render_with_playwright("https://quotes.toscrape.com/js/", wait_selector=".banana")
    print("성공:", len(test))
except Exception as e:
    print("기다리던 요소가 안 나타나 시간 초과/실패:", type(e).__name__)

기다리던 요소가 안 나타나 시간 초과/실패: TypeError


In [12]:
# 예제: AI가 짜줬다고 가정한 '초안' 스크래퍼 — 그런데 셀렉터가 추측이라 틀릴 수 있다
def ai_draft_scraper(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for item in soup.select(".item"):              # ← AI가 추측한 셀렉터 (실제는 .product!)
        name = item.select_one(".title").get_text()
        price = item.select_one(".price").get_text()
        rows.append({"name": name, "price": price})
    return pd.DataFrame(rows)

# [3] 검증 — 그냥 믿지 말고 실행해서 결과를 본다
draft = ai_draft_scraper(RENDERED_HTML)
print("AI 초안 결과 행 수:", len(draft))   # 0 !
print(draft)

AI 초안 결과 행 수: 0
Empty DataFrame
Columns: []
Index: []


In [13]:
# 예제: [4] 수정 — 실제 HTML을 보고 셀렉터를 바로잡는다
def fixed_scraper(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for item in soup.select(".product"):           # .item → .product 로 수정
        name_el = item.select_one(".name")          # .title → .name 으로 수정
        price_el = item.select_one(".price")
        rows.append({
            "name": name_el.get_text(strip=True) if name_el else None,
            "price": price_el.get_text(strip=True) if price_el else None,
        })
    return pd.DataFrame(rows)

fixed = fixed_scraper(RENDERED_HTML)
print("수정 후 결과 행 수:", len(fixed))   # 8
display(fixed.head())

수정 후 결과 행 수: 8


,name,price
0,무선 이어폰,"89,000원"
1,블루투스 스피커,"49,000원"
2,노트북 거치대,"29,000원"
3,기계식 키보드,₩119000
4,USB-C 충전기,19000


In [14]:
# 예제: 검증 체크리스트 2번('결과가 비지 않았나')을 함수로 만들어 적용
def verify_not_empty(df, label):
    ok = len(df) > 0
    print(f"[검증] {label}: {len(df)}행 -> {'통과 OK' if ok else '실패 (셀렉터 의심!)'}")
    return ok

verify_not_empty(ai_draft_scraper(RENDERED_HTML), "AI 초안")
verify_not_empty(fixed_scraper(RENDERED_HTML), "수정본")

[검증] AI 초안: 0행 -> 실패 (셀렉터 의심!)
[검증] 수정본: 8행 -> 통과 OK


True

In [43]:
# 스스로 해보자! (4)
# fixed_scraper를 복사해 category를 추가해보세요.

def fixed_scraper_v2(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for item in soup.select(".product"):
        name_el = item.select_one(".name")
        cat_el = item.select_one(".category")
        price_el = item.select_one(".price")
        rows.append({
            "name": name_el.get_text(strip=True) if name_el else None,
            "category": cat_el.get_text(strip=True) if cat_el else None,
            "price": price_el.get_text(strip=True) if price_el else None,
        })
    return pd.DataFrame(rows)


In [16]:
# [브라우저 설치 필요] — '동작 방식'을 보여주는 예시 (샌드박스 quotes.toscrape.com/login 기준)
# 정의만 합니다. 실제 호출은 브라우저가 설치된 환경에서, 허용된 사이트에 한해서만.
def scrape_with_login(url, username, password):
    """로그인이 필요한 페이지 수집의 '구조'를 보여주는 예시."""
    from playwright.sync_api import sync_playwright
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto(url)
        page.fill("input[name='username']", username)   # 아이디 입력
        page.fill("input[name='password']", password)   # 비번 입력
        page.click("button[type='submit']")             # 로그인 클릭
        page.wait_for_selector(".quote")                # 로그인 후 콘텐츠 대기
        html = page.content()
        browser.close()
    return html

print("로그인 스크래퍼 '정의' 완료 — 호출은 브라우저 설치 + 허용된 사이트에서만.")

로그인 스크래퍼 '정의' 완료 — 호출은 브라우저 설치 + 허용된 사이트에서만.


In [17]:
# [브라우저 설치 필요] — 무한스크롤·팝업 처리 '동작 방식' 예시 (정의만)
def scrape_infinite_scroll(url, rounds=3, pause=1.0):
    """아래로 여러 번 스크롤해 추가 로딩을 유발하는 구조."""
    from playwright.sync_api import sync_playwright
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto(url)
        for _ in range(rounds):
            page.mouse.wheel(0, 4000)     # 아래로 스크롤
            page.wait_for_timeout(int(pause * 1000))  # 새 콘텐츠 로딩 대기(예의 있게)
        html = page.content()
        browser.close()
    return html

def handle_popup(page):
    """팝업(alert/confirm)을 자동으로 닫는 패턴."""
    page.on("dialog", lambda dialog: dialog.dismiss())   # 뜨면 자동 닫기

print("무한스크롤·팝업 처리 함수 '정의' 완료.")

무한스크롤·팝업 처리 함수 '정의' 완료.


In [ ]:
# 스스로 해보자! (5) — 코드 실행이 아니라, 아래 질문에 주석으로 답해보세요.
# 1) 비번 입력 줄: page.fill("input[name='password']", password)
#    콘텐츠 대기 줄: page.wait_for_selector(".quote")
# 2) rounds=100 이면? → 스크롤을 100번 반복하므로, 페이지가 매우 길어지고 로딩 시간이 길어질 수 있습니다. 서버에 부담을 줄 수 있으며, 일부 사이트에서는 과도한 요청으로 인해 차단될 수도 있습니다.
# 3) 다루지 않기로 한 것? → (a) 샌드박스 로그인 구조 이해 (b) 실제 쇼핑몰 로그인 우회 (c) 스크롤로 추가 로딩  B 실제 쇼핑몰 로그인 우회 
print("스스로 해보자 (5)는 판단 연습입니다. 위 주석에 답을 적어보세요.")

스스로 해보자 (5)는 판단 연습입니다. 위 주석에 답을 적어보세요.


In [19]:
# 예제: robots.txt 읽고 판단하기 — 네트워크 없이 '내용 파싱'으로 시연(항상 실행)
from urllib.robotparser import RobotFileParser

# 실제 robots.txt가 이렇게 생겼다고 가정합니다.
robots_text = """User-agent: *
Disallow: /admin/
Disallow: /private/
Disallow: /cart
Allow: /
Crawl-delay: 1
"""

rp = RobotFileParser()
rp.parse(robots_text.splitlines())   # 파일 내용을 직접 파싱(네트워크 불필요)

for path in ["/products", "/admin/users", "/private/keys", "/cart"]:
    allowed = rp.can_fetch("*", path)
    print(f"{path:<16} 수집 가능? → {allowed}")

/products        수집 가능? → True
/admin/users     수집 가능? → False
/private/keys    수집 가능? → False
/cart            수집 가능? → False


In [49]:
# [네트워크 필요] 실제 사이트의 robots.txt 확인 — 안 되면 위 로컬 시연으로 충분
try:
    rp_live = RobotFileParser()
    rp_live.set_url("https://naver.com/robots.txt")
    rp_live.read()                      # 네트워크로 robots.txt 가져오기
    can = rp_live.can_fetch("*", "https://naver.com/catalogue/")
    print("naver.com /catalogue 수집 가능? →", can)
except Exception as e:
    print("네트워크 불가 →", type(e).__name__)
    print("→ 괜찮습니다. 위 셀의 로컬 시연으로 robots.txt 판단법은 이미 익혔습니다.")

naver.com /catalogue 수집 가능? → False


In [50]:
# 예제: robots.txt의 Crawl-delay(요청 간격) 읽기
delay = rp.crawl_delay("*")
print("이 사이트가 요구하는 요청 간격(Crawl-delay):", delay, "초")
print(f"-> 그렇다면 요청 사이에 최소 {delay or 1}초는 쉬어주는 것이 예의입니다.")

이 사이트가 요구하는 요청 간격(Crawl-delay): 1 초
-> 그렇다면 요청 사이에 최소 1초는 쉬어주는 것이 예의입니다.


In [51]:
# 예제: 요청 사이 간격 두기(rate limiting) 시뮬레이션 — 서버에 예의를 갖추는 패턴
pages = ["/page/1", "/page/2", "/page/3"]
headers = {"User-Agent": "MoodleMarket-Edu-Scraper/1.0 (학습용)"}  # 자신을 밝히기

for page_path in pages:
    # 실제로는 여기서 requests.get(base_url + page_path, headers=headers)
    print(f"요청 → {page_path}  (User-Agent: {headers['User-Agent']})")
    time.sleep(0.5)   # 요청 사이 간격 (robots.txt의 Crawl-delay 존중)

print("\n→ 요청마다 간격을 두고, User-Agent로 신원을 밝혔습니다. 서버 부하를 줄이는 기본 예의입니다.")

요청 → /page/1  (User-Agent: MoodleMarket-Edu-Scraper/1.0 (학습용))
요청 → /page/2  (User-Agent: MoodleMarket-Edu-Scraper/1.0 (학습용))
요청 → /page/3  (User-Agent: MoodleMarket-Edu-Scraper/1.0 (학습용))

→ 요청마다 간격을 두고, User-Agent로 신원을 밝혔습니다. 서버 부하를 줄이는 기본 예의입니다.


In [52]:
# 스스로 해보자! (6)
# 아래 주석(#)을 지우고 빈칸(___)을 채워보세요.

robots_text2 = robots_text + "Allow: /search\n"     # 규칙 추가
rp2 = RobotFileParser()
rp2.parse(robots_text2.splitlines())
print("/search 수집 가능? →", rp2.can_fetch("*", "/search"))
print("/products/123 수집 가능? →", rp2.can_fetch("*", "/products/123"))

/search 수집 가능? → True
/products/123 수집 가능? → True


In [24]:
# 종합 실습용 페이지 — 두 버전 HTML(네트워크 불필요). 구조가 Part 1~5와 다릅니다(새 셀렉터!).
BEST_STATIC_HTML = """<!DOCTYPE html><html lang="ko"><head><title>모두마켓 베스트 도서</title></head>
<body><h1>이주의 베스트 도서</h1>
  <ul id="book-list"><li class="loading">불러오는 중...</li></ul>
  <script src="/static/render_books.js"></script>
</body></html>"""

BEST_RENDERED_HTML = """<!DOCTYPE html><html lang="ko"><head><title>모두마켓 베스트 도서</title></head>
<body><h1>이주의 베스트 도서</h1>
  <ul id="book-list">
    <li class="book" data-isbn="978-1">
      <h3 class="title">파이썬으로 시작하는 데이터 분석</h3>
      <span class="author">김데이터</span><span class="price">28,000원</span><span class="rating">4.8</span>
    </li>
    <li class="book" data-isbn="978-2">
      <h3 class="title">  모두를 위한 통계학  </h3>
      <span class="author">이통계</span><span class="price">32,000원</span><span class="rating">4.6</span>
    </li>
    <li class="book" data-isbn="978-3">
      <h3 class="title">웹 스크래핑 실전</h3>
      <span class="author">박수집</span><span class="price">₩26000</span>
    </li>
    <li class="book" data-isbn="978-4">
      <h3 class="title">머신러닝 입문</h3>
      <span class="author">최모델</span><span class="price">35000</span><span class="rating">4.9</span>
    </li>
    <li class="book" data-isbn="978-5">
      <h3 class="title">SQL 첫걸음</h3>
      <span class="author">정쿼리</span><span class="price">24,000원</span><span class="rating">4.5</span>
    </li>
  </ul>
  <script src="/static/render_books.js"></script>
</body></html>"""

print("종합 실습용 HTML 준비 완료 (정적/렌더링 2종)")

종합 실습용 HTML 준비 완료 (정적/렌더링 2종)


In [25]:
# 시나리오 1 — 정적 vs 렌더링 비교로 동적 페이지임을 확인
soup_static_b = BeautifulSoup(BEST_STATIC_HTML, "html.parser")
soup_rendered_b = BeautifulSoup(BEST_RENDERED_HTML, "html.parser")

print("정적 HTML의 .book 수  :", len(soup_static_b.select(".book")))      # 0 → 동적!
print("렌더링 HTML의 .book 수:", len(soup_rendered_b.select(".book")))   # 5
print("\n판별 결과: 소스엔 없고 화면엔 있으므로 → 동적 페이지 (Playwright 필요)")

정적 HTML의 .book 수  : 0
렌더링 HTML의 .book 수: 5

판별 결과: 소스엔 없고 화면엔 있으므로 → 동적 페이지 (Playwright 필요)


In [26]:
# 시나리오 2 — (가정) AI가 준 초안: 셀렉터가 틀렸다
def ai_book_draft(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for li in soup.select("li.item"):          # ← 틀림: 실제는 li.book
        rows.append({"title": li.select_one(".title").get_text()})
    return pd.DataFrame(rows)

print("[검증] AI 초안 결과 행 수:", len(ai_book_draft(BEST_RENDERED_HTML)))   # 0 → 셀렉터 의심!

# 수정: 실제 HTML을 보고 li.book / 올바른 셀렉터로 고친다
def book_scraper(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for li in soup.select("li.book"):
        rating_el = li.select_one(".rating")           # 없을 수 있음(결측!)
        rows.append({
            "isbn": li.get("data-isbn"),
            "title": li.select_one(".title").get_text(),
            "author": li.select_one(".author").get_text(strip=True),
            "price_raw": li.select_one(".price").get_text(strip=True),
            "rating": rating_el.get_text(strip=True) if rating_el else None,
        })
    return pd.DataFrame(rows)

books = book_scraper(BEST_RENDERED_HTML)
print("[수정 후] 행 수:", len(books))
display(books)

[검증] AI 초안 결과 행 수: 0
[수정 후] 행 수: 5


,isbn,title,author,price_raw,rating
0,978-1,파이썬으로 시작하는 데이터 분석,김데이터,"28,000원",4.8
1,978-2,모두를 위한 통계학,이통계,"32,000원",4.6
2,978-3,웹 스크래핑 실전,박수집,₩26000,NaN
3,978-4,머신러닝 입문,최모델,35000,4.9
4,978-5,SQL 첫걸음,정쿼리,"24,000원",4.5


In [27]:
# 시나리오 3 — 정제(공백·결측·가격) + robots.txt 검토
books_clean = books.copy()
books_clean["title"] = books_clean["title"].str.strip()                 # 공백 제거(D+005)
books_clean["rating"] = books_clean["rating"].fillna("평점없음")          # 결측 대체(D+003)
books_clean["price"] = books_clean["price_raw"].str.replace(r"[^0-9]", "", regex=True).astype(int)  # 숫자만(D+005)
books_final = books_clean[["isbn", "title", "author", "price", "rating"]]

print("[정제 완료]")
display(books_final)

# robots.txt 윤리 검토(로컬 시연)
rp_b = RobotFileParser()
rp_b.parse(["User-agent: *", "Allow: /", "Disallow: /admin/", "Crawl-delay: 1"])
print("이 페이지(/best-books) 수집 가능? →", rp_b.can_fetch("*", "/best-books"))
print("평균 도서 가격:", int(books_final["price"].mean()), "원")

[정제 완료]


,isbn,title,author,price,rating
0,978-1,파이썬으로 시작하는 데이터 분석,김데이터,28000,4.8
1,978-2,모두를 위한 통계학,이통계,32000,4.6
2,978-3,웹 스크래핑 실전,박수집,26000,평점없음
3,978-4,머신러닝 입문,최모델,35000,4.9
4,978-5,SQL 첫걸음,정쿼리,24000,4.5


이 페이지(/best-books) 수집 가능? → True
평균 도서 가격: 29000 원


In [28]:
# 종합 — 정리된 데이터로 간단 분석: 평점 있는 도서를 평점 높은 순으로
rated_books = books_final[books_final["rating"] != "평점없음"].copy()
rated_books["rating_num"] = rated_books["rating"].astype(float)
top_books = rated_books.sort_values("rating_num", ascending=False)
print("평점 높은 순 베스트 도서:")
display(top_books[["title", "author", "price", "rating"]])

평점 높은 순 베스트 도서:


,title,author,price,rating
3,머신러닝 입문,최모델,35000,4.9
0,파이썬으로 시작하는 데이터 분석,김데이터,28000,4.8
1,모두를 위한 통계학,이통계,32000,4.6
4,SQL 첫걸음,정쿼리,24000,4.5


In [29]:
# 코드 퀴즈 — 모범 답안
# rating이 '평점없음'이 아닌(=실제 평점이 있는) 도서만 필터
rated = books_final[books_final["rating"] != "평점없음"]

print("평점이 있는 도서 수:", len(rated))
print("평점 있는 도서 평균 가격:", int(rated["price"].mean()), "원")
display(rated[["title", "price", "rating"]])

평점이 있는 도서 수: 4
평점 있는 도서 평균 가격: 29750 원


,title,price,rating
0,파이썬으로 시작하는 데이터 분석,28000,4.8
1,모두를 위한 통계학,32000,4.6
3,머신러닝 입문,35000,4.9
4,SQL 첫걸음,24000,4.5
